# Section 7-11: Differential Privacy Track

## Overview

This notebook covers the Differential Privacy (DP) track of the AI Privacy module. We'll explore:

- **Section 7:** Differential Privacy Fundamentals
- **Section 8:** Opacus Library Setup
- **Section 9:** DP-SGD Setup Overview
- **Section 10:** Training Models and Measuring Privacy
- **Section 11:** Analyzing the Privacy-Utility Tradeoff
- **Section 12:** DP-SGD Challenge (separate)

### Key Concepts

1. **Differential Privacy (DP):** A formal framework for quantifying privacy loss
2. **DP-SGD:** Modified training procedure with gradient clipping and noise addition
3. **Privacy Budget (ε, δ):** Parameters controlling privacy strength
4. **Privacy-Utility Tradeoff:** Balancing privacy protection with model accuracy

### Dataset

We use CIFAR-10:
- 50,000 training images (10 classes)
- 10,000 test images
- Image size: 32×32 pixels
- Classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck

## Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## CIFAR-10 Dataset Preparation

In [ ]:
# Define transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Create data loaders
batch_size = 256
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")
print(f"Number of classes: {len(trainset.classes)}")
print(f"Classes: {trainset.classes}")

## CNN Architecture for CIFAR-10

In [ ]:
class CIFAR10_CNN(nn.Module):
    """Simple CNN for CIFAR-10 classification"""
    
    def __init__(self, num_classes=10):
        super(CIFAR10_CNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Pooling and batch normalization
        self.pool = nn.MaxPool2d(2, 2)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Fully connected layers
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Block 1: Conv -> BN -> ReLU -> Pool
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool(x)  # 32 -> 16
        
        # Block 2: Conv -> BN -> ReLU -> Pool
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)  # 16 -> 8
        
        # Block 3: Conv -> BN -> ReLU -> Pool
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool(x)  # 8 -> 4
        
        # Flatten and fully connected layers
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.fc3(x)
        return x

# Create model
model = CIFAR10_CNN(num_classes=10).to(device)
print(f"Model created and moved to {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Function

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy

def test(model, test_loader, criterion, device):
    """Test the model"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="Testing", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    
    return avg_loss, accuracy

print("Training and test functions defined")

## Differential Privacy Concepts Visualization

In [ ]:
# Visualize the differential privacy concepts
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Differential Privacy Fundamentals', fontsize=16, fontweight='bold')

# 1. Privacy Budget (epsilon) impact
epsilon_values = np.array([0.1, 0.5, 1, 3, 10, 100])
exp_epsilon = np.exp(epsilon_values)

axes[0, 0].bar(range(len(epsilon_values)), exp_epsilon, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xticks(range(len(epsilon_values)))
axes[0, 0].set_xticklabels([f'ε={e}' for e in epsilon_values])
axes[0, 0].set_ylabel('e^ε (Multiplicative Bound)', fontsize=11)
axes[0, 0].set_title('Privacy Budget vs Output Likelihood Bound\n(Lower ε = Stronger Privacy)', fontsize=11)
axes[0, 0].set_yscale('log')
axes[0, 0].grid(axis='y', alpha=0.3)

# 2. Gradient Clipping Visualization
grad_norms = np.array([0.3, 0.7, 0.95, 1.2, 1.5, 2.0, 3.5, 5.2])
max_grad_norm = 1.0
clipped_norms = np.minimum(grad_norms, max_grad_norm)

x_pos = np.arange(len(grad_norms))
axes[0, 1].bar(x_pos, grad_norms, label='Original Gradient Norm', alpha=0.6, color='coral', edgecolor='black')
axes[0, 1].bar(x_pos, clipped_norms, label='Clipped Gradient Norm', alpha=0.8, color='darkred', edgecolor='black')
axes[0, 1].axhline(y=max_grad_norm, color='red', linestyle='--', linewidth=2, label=f'Clipping Threshold={max_grad_norm}')
axes[0, 1].set_ylabel('Norm Value', fontsize=11)
axes[0, 1].set_xlabel('Sample Index', fontsize=11)
axes[0, 1].set_title('Gradient Clipping: Bounding Sensitivity', fontsize=11)
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)

# 3. Noise Addition for Different Privacy Levels
epsilon_configs = [10, 3, 1]
noise_multipliers = [1.2, 3.8, 10.0]  # Approximate values for CIFAR-10
max_grad_norm_val = 1.0
noise_stds = [max_grad_norm_val * nm for nm in noise_multipliers]

colors = ['green', 'orange', 'red']
x_noise = np.linspace(-5, 5, 1000)

for i, (eps, noise_std, color) in enumerate(zip(epsilon_configs, noise_stds, colors)):
    noise_dist = (1 / (noise_std * np.sqrt(2 * np.pi))) * np.exp(-(x_noise ** 2) / (2 * noise_std ** 2))
    axes[1, 0].plot(x_noise, noise_dist, label=f'ε={eps} (σ={noise_std:.1f})', linewidth=2.5, color=color)

axes[1, 0].set_xlabel('Noise Value', fontsize=11)
axes[1, 0].set_ylabel('Probability Density', fontsize=11)
axes[1, 0].set_title('Gaussian Noise Addition\n(Higher Privacy = More Noise)', fontsize=11)
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(alpha=0.3)

# 4. Privacy-Utility Tradeoff Illustration
epsilon_range = np.array([0.5, 1, 2, 3, 5, 8, 10, 15, 20])
# Simulated relationship: higher epsilon -> less privacy loss -> higher utility
utility_scores = 100 * (1 - np.exp(-0.3 * epsilon_range))
privacy_strength = 100 / (1 + 0.15 * epsilon_range)

axes[1, 1].plot(epsilon_range, utility_scores, 'o-', linewidth=2.5, markersize=8, label='Model Utility (Accuracy)', color='green')
axes[1, 1].plot(epsilon_range, privacy_strength, 's-', linewidth=2.5, markersize=8, label='Privacy Strength', color='red')
axes[1, 1].axvline(x=3, color='blue', linestyle='--', alpha=0.7, linewidth=2, label='ε=3 (Strong Privacy)')
axes[1, 1].axvline(x=10, color='orange', linestyle='--', alpha=0.7, linewidth=2, label='ε=10 (Modest Privacy)')
axes[1, 1].set_xlabel('Privacy Budget (ε)', fontsize=11)
axes[1, 1].set_ylabel('Score (%)', fontsize=11)
axes[1, 1].set_title('Privacy-Utility Tradeoff', fontsize=11)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_ylim([0, 105])

plt.tight_layout()
plt.savefig('./output/07_dp_fundamentals_concepts.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nDifferential Privacy Concepts Visualized:")
print("1. Privacy Budget (ε) Impact: Shows how privacy strength increases with smaller ε")
print("2. Gradient Clipping: Bounds individual sample influence on gradients")
print("3. Gaussian Noise Addition: Adds noise calibrated to privacy budget")
print("4. Privacy-Utility Tradeoff: Stronger privacy requires sacrificing some accuracy")

## Privacy Accounting: Budget Composition Across Training Steps

In [ ]:
# Privacy accounting visualization
def compute_privacy_budget(num_epochs, batch_size, dataset_size, target_epsilon, delta=1e-5):
    """
    Estimate noise multiplier needed to achieve target epsilon
    Using simplified RDP accounting
    """
    num_steps = (dataset_size / batch_size) * num_epochs
    return num_steps

# Parameters
dataset_size = 50000  # CIFAR-10 training set
batch_size = 256
num_epochs = 10
target_epsilons = [10, 3, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Privacy Budget Composition Across Training', fontsize=14, fontweight='bold')

# Calculate total steps
total_steps = compute_privacy_budget(num_epochs, batch_size, dataset_size, 10)
epochs_per_step = np.arange(1, num_epochs + 1)
steps_per_epoch = dataset_size / batch_size

# 1. Cumulative privacy loss with different epsilon values
for target_eps in target_epsilons:
    # Simplified: each step uses epsilon_per_step
    epsilon_per_step = target_eps / num_epochs
    cumulative_eps = epochs_per_step * epsilon_per_step
    
    axes[0].plot(epochs_per_step, cumulative_eps, 'o-', linewidth=2.5, markersize=8, 
                 label=f'Target ε={target_eps} (ε/step={epsilon_per_step:.3f})')

axes[0].set_xlabel('Training Epoch', fontsize=11)
axes[0].set_ylabel('Cumulative Privacy Loss (ε)', fontsize=11)
axes[0].set_title('How Privacy Budget Accumulates Over Training Steps', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# 2. Required noise multiplier for different configurations
epsilon_values = np.array([1, 3, 5, 8, 10])
noise_multipliers_est = [10.0, 3.8, 2.5, 1.5, 1.2]  # Estimated for CIFAR-10, batch_size=256

axes[1].plot(epsilon_values, noise_multipliers_est, 'o-', linewidth=2.5, markersize=10, 
             color='darkblue', label='Estimated Noise Multiplier')
axes[1].fill_between(epsilon_values, 0, noise_multipliers_est, alpha=0.3, color='lightblue')

# Annotations
for eps, nm in zip(epsilon_values, noise_multipliers_est):
    axes[1].annotate(f'σ={nm:.1f}', xy=(eps, nm), xytext=(5, 5), 
                    textcoords='offset points', fontsize=9, fontweight='bold')

axes[1].set_xlabel('Privacy Budget (ε)', fontsize=11)
axes[1].set_ylabel('Noise Multiplier (σ)', fontsize=11)
axes[1].set_title('Privacy vs Required Noise\n(Higher Privacy = Higher Noise = Harder Learning)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('./output/07_privacy_accounting.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPrivacy Accounting Summary (CIFAR-10):")
print(f"  Dataset size: {dataset_size:,}")
print(f"  Batch size: {batch_size}")
print(f"  Training epochs: {num_epochs}")
print(f"  Total training steps: {int(total_steps * num_epochs):,}")
print(f"  Steps per epoch: {int(steps_per_epoch):.0f}")
print(f"\nNoise Multiplier Requirements:")
for eps, nm in zip(epsilon_values, noise_multipliers_est):
    print(f"  ε = {eps:2d}: noise_multiplier ≈ {nm:4.1f}")

## Section 7 Summary

### Key Takeaways

1. **Differential Privacy Definition:** (ε, δ)-DP provides formal privacy guarantees by bounding how much any single record can affect the algorithm's output distribution.

2. **Privacy Parameters:**
   - **ε (epsilon):** Privacy budget - lower values = stronger privacy
     - ε = 1: Very strong privacy
     - ε = 3: Strong privacy (academic standard)
     - ε = 10: Modest privacy (practical baseline)
   - **δ (delta):** Failure probability - typically set to 10^-5 for large datasets

3. **DP-SGD Mechanism:** Three components:
   - **Gradient Clipping:** Bounds L2 norm to `max_grad_norm` (typically 1.0)
   - **Noise Addition:** Gaussian noise with σ = max_grad_norm × noise_multiplier
   - **Privacy Composition:** Tracks cumulative privacy loss across training steps

4. **Privacy-Utility Tradeoff:**
   - Stronger privacy (lower ε) requires higher noise
   - Higher noise makes optimization harder
   - Results in lower model accuracy

5. **Privacy Accounting:**
   - Privacy loss accumulates with each training step
   - Advanced composition (RDP) provides tighter bounds than naive addition
   - Opacus automatically computes required noise multiplier

### Next Steps

In the following sections, we'll:
- Install and configure Opacus (Section 8)
- Set up DP-SGD training (Section 9)
- Train privacy-preserving models (Section 10)
- Analyze privacy-utility tradeoffs (Section 11)
- Complete DP-SGD challenge (Section 12)

# Section 8: The Opacus Library

## Overview

Opacus is a PyTorch library that implements DP-SGD. It automatically handles:
- Per-sample gradient computation
- Gradient clipping and noise addition
- Privacy accounting and cumulative budget tracking

### Key Components
1. **PrivacyEngine:** Wraps optimizer and data loader, modifies gradient computation
2. **Privacy Accountant:** Tracks cumulative privacy expenditure (ε, δ)
3. **Module Validator:** Checks and fixes layer compatibility

## Per-Sample Gradients and Architecture Constraints

Standard PyTorch backprop computes average gradient across batch. DP-SGD requires **per-sample gradients** where each sample's gradient is computed separately, clipped, and noised.

### Computational Cost
- Memory: Increases proportionally to batch size
- Speed: ~2-5x slower than standard SGD
- For batch of 256: compute 256 gradient tensors per parameter (instead of 1)

In [ ]:
# Layer Compatibility for DP-SGD
import torch.nn as nn
import pandas as pd

# Create compatibility table
compatibility_data = {
    'Layer Type': ['BatchNorm2d', 'GroupNorm', 'LayerNorm', 'InstanceNorm', 'Dropout', 'Conv2d', 'Linear', 'Custom Layer'],
    'DP-SGD Compatible': ['❌ NO', '✅ YES', '✅ YES', '✅ YES', '✅ YES', '✅ YES', '✅ YES', 'Depends'],
    'Reason': [
        'Couples sample gradients via batch statistics',
        'Normalizes within samples independently',
        'Normalizes within samples independently',
        'Operates on individual samples',
        'Compatible - no state sharing',
        'Compatible - per-sample gradients available',
        'Compatible - per-sample gradients available',
        'Depends on sample coupling'
    ]
}

df_compat = pd.DataFrame(compatibility_data)
print("\\n" + "="*100)
print("LAYER COMPATIBILITY FOR DP-SGD")
print("="*100)
print(df_compat.to_string(index=False))
print("\\nKey Rule: Layers that couple sample gradients across batch dimension are incompatible.\\n")

# Show how to replace BatchNorm with GroupNorm
print("\\n" + "="*100)
print("FIXING INCOMPATIBILITIES")
print("="*100)
print("""
# Problem: BatchNorm couples samples
old_layer = nn.BatchNorm2d(64)

# Solution: Replace with GroupNorm
# For channel-wise normalization similar to BatchNorm:
new_layer = nn.GroupNorm(num_groups=64, num_channels=64)

# For feature-wise normalization:
new_layer = nn.GroupNorm(num_groups=1, num_channels=64)
""")

## Privacy Amplification by Subsampling

Opacus uses **Poisson subsampling**: each example is included in batch independently with probability q = batch_size / dataset_size.

### Benefits
- Attacker doesn't know which samples were in any given batch
- Effective privacy cost per step is reduced
- Amplifies privacy without additional noise

### Example (CIFAR-10)
- Batch size: 256
- Dataset size: 50,000
- Sampling rate: q = 256/50,000 = 0.00512 (0.512%)
- Each sample has 0.512% independent chance per batch

In [ ]:
# Privacy Amplification by Subsampling Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Privacy Amplification by Poisson Subsampling', fontsize=14, fontweight='bold')

# 1. Sampling rate vs batch size
dataset_size = 50000
batch_sizes = np.array([32, 64, 128, 256, 512, 1024])
sampling_rates = batch_sizes / dataset_size * 100

axes[0].bar(range(len(batch_sizes)), sampling_rates, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xticks(range(len(batch_sizes)))
axes[0].set_xticklabels([f'B={b}' for b in batch_sizes])
axes[0].set_ylabel('Sampling Rate q (%)', fontsize=11)
axes[0].set_title('Batch Size vs Sampling Rate\n(Lower q = More Privacy Amplification)', fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

for i, (bs, sr) in enumerate(zip(batch_sizes, sampling_rates)):
    axes[0].text(i, sr + 0.01, f'{sr:.2f}%', ha='center', va='bottom', fontsize=9)

# 2. Effective privacy cost reduction
# Simplified: amplification factor approximately 1/sqrt(q)
amplification_factors = 1 / np.sqrt(sampling_rates / 100 + 0.0001)

axes[1].plot(batch_sizes, amplification_factors, 'o-', linewidth=2.5, markersize=10, 
             color='darkgreen', label='Amplification Factor')
axes[1].fill_between(batch_sizes, 1, amplification_factors, alpha=0.3, color='lightgreen')
axes[1].set_xlabel('Batch Size', fontsize=11)
axes[1].set_ylabel('Privacy Amplification Factor', fontsize=11)
axes[1].set_title('Privacy Amplification by Subsampling\n(Higher = Stronger Privacy)', fontsize=11)
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('./output/08_privacy_amplification.png', dpi=300, bbox_inches='tight')
plt.show()

print("\\nPoisson Subsampling Analysis (CIFAR-10):") 
print("="*60)
for bs, sr, amp in zip(batch_sizes, sampling_rates, amplification_factors):
    print(f"Batch size {bs:4d}: q = {sr:5.3f}% | Amplification ≈ {amp:6.2f}x")

## Tuning DP-SGD Hyperparameters

### Clipping Threshold (max_grad_norm)
- **Too aggressive** (0.1): Gradients truncated, model starves of learning signal
- **Too permissive** (10.0): Waste privacy budget on unused headroom
- **Calibration:** Compute 75th percentile of gradient norms without privacy
- **CIFAR-10 CNN:** Gradient norms typically 0.5-5.0, so **max_grad_norm = 1.0** is reasonable

### Batch Size
- **Smaller** (<64): Lower sampling rate → better privacy but high gradient variance
- **Moderate** (128-512): Good balance for convergence and privacy
- **Larger** (>512): Weaker privacy amplification but fewer steps

### Learning Rate
- Start with **baseline learning rate** (privacy doesn't change optimal LR much)
- Noise already reduces gradient signal, so more reduction may not help
- If divergence: lower LR to stabilize optimization
- Some practitioners find **no momentum** works better with DP-SGD

### Number of Epochs
- **More epochs:** Better convergence but higher total privacy cost
- **Trade-off:** More epochs require higher noise per step to maintain target ε
- **Strategy:** Early stopping when convergence plateaus

### Failure Probability (delta)
- Must satisfy: **δ < 1/n** where n = dataset size
- CIFAR-10: n = 50,000, so δ = 10^-5 = 1/100,000 ✅
- Ensures privacy guarantee doesn't degrade

In [ ]:
# Hyperparameter Tuning: Trade-offs Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('DP-SGD Hyperparameter Tuning', fontsize=14, fontweight='bold')

# 1. Clipping Threshold Impact
clipping_thresholds = np.array([0.1, 0.5, 1.0, 2.0, 5.0, 10.0])
info_retention = 1 - np.exp(-0.5 * clipping_thresholds)  # Sigmoid-like
learning_efficiency = info_retention * 100

axes[0, 0].plot(clipping_thresholds, learning_efficiency, 'o-', linewidth=2.5, markersize=10, 
                color='darkblue', label='Learning Efficiency')
axes[0, 0].axvline(x=1.0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Recommended (1.0)')
axes[0, 0].set_xlabel('Clipping Threshold (max_grad_norm)', fontsize=11)
axes[0, 0].set_ylabel('Gradient Information Retained (%)', fontsize=11)
axes[0, 0].set_title('Clipping Threshold vs Learning Efficiency', fontsize=11)
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(alpha=0.3)
axes[0, 0].set_ylim([0, 105])

# 2. Batch Size Trade-off
batch_sizes_hp = np.array([32, 64, 128, 256, 512])
privacy_amplification = 1 / np.sqrt((batch_sizes_hp / 50000) + 0.0001)
gradient_variance = 1 / np.sqrt(batch_sizes_hp)  # Normalized

ax2_twin = axes[0, 1].twinx()
line1 = axes[0, 1].plot(batch_sizes_hp, privacy_amplification, 'o-', linewidth=2.5, markersize=10,
                         color='green', label='Privacy Amplification')
line2 = ax2_twin.plot(batch_sizes_hp, gradient_variance * 100, 's-', linewidth=2.5, markersize=10,
                       color='red', label='Gradient Variance (normalized)')

axes[0, 1].set_xlabel('Batch Size', fontsize=11)
axes[0, 1].set_ylabel('Privacy Amplification', fontsize=11, color='green')
ax2_twin.set_ylabel('Gradient Variance (%)', fontsize=11, color='red')
axes[0, 1].set_title('Batch Size Trade-offs', fontsize=11)
axes[0, 1].tick_params(axis='y', labelcolor='green')
ax2_twin.tick_params(axis='y', labelcolor='red')
axes[0, 1].axvline(x=256, color='blue', linestyle='--', linewidth=2, alpha=0.5, label='Typical (256)')
lines = line1 + line2
labels = [l.get_label() for l in lines]
axes[0, 1].legend(lines, labels, loc='upper right', fontsize=9)
axes[0, 1].grid(alpha=0.3)

# 3. Epochs vs Privacy Cost
epochs_range = np.array([1, 5, 10, 20, 30, 50])
target_epsilon = 3
noise_per_step = target_epsilon / epochs_range
final_accuracy_est = 85 - 5 * np.log(noise_per_step + 0.1)

axes[1, 0].plot(epochs_range, final_accuracy_est, 'o-', linewidth=2.5, markersize=10,
                color='purple', label='Estimated Final Accuracy')
axes[1, 0].set_xlabel('Number of Epochs', fontsize=11)
axes[1, 0].set_ylabel('Test Accuracy (%)', fontsize=11)
axes[1, 0].set_title('Epochs vs Model Accuracy\n(Target ε=3)', fontsize=11)
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend(fontsize=10)
axes[1, 0].set_ylim([60, 95])

# 4. Delta Constraint Visualization
dataset_sizes = np.array([1000, 10000, 50000, 100000, 1000000])
max_delta_allowed = 1 / dataset_sizes
recommended_delta = 1e-5 * np.ones_like(dataset_sizes)

axes[1, 1].bar(range(len(dataset_sizes)), max_delta_allowed, alpha=0.6, label='Max δ Allowed (1/n)', 
               color='steelblue', edgecolor='black', width=0.4)
axes[1, 1].bar(np.arange(len(dataset_sizes)) + 0.4, recommended_delta, alpha=0.6, label='Typical δ (1e-5)',
               color='orange', edgecolor='black', width=0.4)
axes[1, 1].set_xticks(np.arange(len(dataset_sizes)) + 0.2)
axes[1, 1].set_xticklabels([f'{n//1000}K' if n >= 1000 else str(n) for n in dataset_sizes])
axes[1, 1].set_ylabel('δ (log scale)', fontsize=11)
axes[1, 1].set_xlabel('Dataset Size (n)', fontsize=11)
axes[1, 1].set_yscale('log')
axes[1, 1].set_title('Delta Constraint: δ < 1/n', fontsize=11)
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./output/08_hyperparameter_tuning.png', dpi=300, bbox_inches='tight')
plt.show()

print("\\n" + "="*70)
print("HYPERPARAMETER TUNING SUMMARY")
print("="*70)
print(f"Clipping Threshold:  max_grad_norm = 1.0 (for CIFAR-10)")
print(f"Batch Size:          256 (good balance)")
print(f"Learning Rate:       Start with baseline (~0.1)")
print(f"Epochs:              10 (adjust based on convergence)")
print(f"Delta:               1e-5 (satisfies δ < 1/50000 for CIFAR-10)")
print("="*70)

## Section 8 Summary: Opacus Library

### Key Takeaways

1. **Opacus Components:**
   - **PrivacyEngine:** Wraps optimizer, computes per-sample gradients
   - **Privacy Accountant:** Tracks cumulative ε and δ expenditure
   - **Module Validator:** Checks layer compatibility and suggests fixes

2. **Per-Sample Gradients:**
   - Required for DP-SGD but computationally expensive
   - Increases memory: proportional to batch size
   - Slows training: ~2-5x compared to standard SGD

3. **Layer Compatibility:**
   - ❌ **Incompatible:** BatchNorm (couples samples across batch)
   - ✅ **Compatible:** GroupNorm, LayerNorm, InstanceNorm, Conv2d, Linear, Dropout
   - **Fix:** Replace BatchNorm with GroupNorm (groups = channels for similar effect)

4. **Privacy Amplification:**
   - **Poisson subsampling** provides amplification without extra noise
   - Sampling rate q = batch_size / dataset_size
   - CIFAR-10: q ≈ 0.5% means 0.5% independent inclusion chance per sample
   - Lower q = stronger amplification = better privacy

5. **Hyperparameter Tuning:**
   - **max_grad_norm:** Calibrate to 75th percentile of gradient norms (CIFAR-10: 1.0)
   - **Batch size:** 128-512 range for good balance
   - **Learning rate:** Start with baseline (no special adjustment needed)
   - **Epochs:** Trade-off between convergence and privacy budget
   - **Delta:** Ensure δ < 1/n (CIFAR-10: δ = 1e-5 ✅)

### Next Section
In Section 9, we'll set up DP-SGD training using Opacus, replacing BatchNorm with GroupNorm and implementing the PrivacyEngine.